In [ ]:
# !pip install --upgrade pip

# ember github for reference 
# !git clone https://github.com/FutureComputing4AI/EMBER2024.git
# %pip install ./EMBER2024

%pip install pandas
%pip install altair

# thrember dependencies
# !pip uninstall -y signify
# %pip install "signify==0.7.1"

# might need libomp installed:
# !brew install libomp

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [altair]2m7/8 [altair]ema]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# imports
import os
import thrember
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
# import lightgbm as lgb
import polars as pl
import altair as alt
# from sklearn.metrics import roc_auc_score, roc_curve
# alt.renderers.enable('default')

In [8]:
DATA_DIR = os.path.abspath("ember2024_data")
os.makedirs(DATA_DIR, exist_ok=True)

In [29]:
# download train and test files for small formats 
SMALL_FORMATS = ["Dot_Net", "APK", "PDF", "ELF"]
SPLITS = ["train", "test"]

for ft in SMALL_FORMATS:
    for split in SPLITS:
        print(f"Downloading file_type={ft}, split={split} ...")
        thrember.download_dataset(DATA_DIR, file_type=ft, split=split)

print("Done. Check du -sh on", DATA_DIR, "before continuing if you want to confirm size.")

Unzipping...
Unzipping...
Unzipping...
Unzipping...
Unzipping...
Unzipping...
Unzipping...
Unzipping...
Done. Check du -sh on /Users/sophieliu/Desktop/CS projects/summer 26/AI4ALL-Project---Malware-Classification-/ember2024_data before continuing if you want to confirm size.


In [15]:
thrember.download_dataset(DATA_DIR, split="challenge")

In [ ]:
# confirm data dir exists and list contents
print("DATA_DIR:", DATA_DIR)
print("Absolute path:", os.path.abspath(DATA_DIR))
print("Exists:", os.path.exists(DATA_DIR))
print("Contents:", os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "Directory does not exist")

DATA_DIR: /Users/sophieliu/Desktop/CS projects/summer 26/AI4ALL-Project---Malware-Classification-/ember2024_data
Absolute path: /Users/sophieliu/Desktop/CS projects/summer 26/AI4ALL-Project---Malware-Classification-/ember2024_data
Exists: True
Contents: ['2024-07-14_2024-07-20_challenge_malicious.jsonl', '2024-06-16_2024-06-22_Dot_Net_train.jsonl', '2024-05-26_2024-06-01_challenge_malicious.jsonl', '2024-06-23_2024-06-29_challenge_malicious.jsonl', '2024-10-13_2024-10-19_Dot_Net_test.jsonl', '2024-03-10_2024-03-16_Dot_Net_train.jsonl', '2024-09-01_2024-09-07_challenge_malicious.jsonl', '2024-09-29_2024-10-05_challenge_malicious.jsonl', '2024-04-07_2024-04-13_Dot_Net_train.jsonl', '2024-04-14_2024-04-20_challenge_malicious.jsonl', '2023-10-29_2023-11-04_Dot_Net_train.jsonl', '2024-06-02_2024-06-08_challenge_malicious.jsonl', '2024-11-10_2024-11-16_Dot_Net_test.jsonl', '2023-10-22_2023-10-28_challenge_malicious.jsonl', '2024-01-14_2024-01-20_Dot_Net_train.jsonl', '2024-07-28_2024-08-03_D

In [30]:
# split the data into train, test, and challenge sets
train_df, test_df, challenge_df = thrember.read_metadata(DATA_DIR)

In [31]:
# Add a 'week' column to the dataframe
plotdf = pl.concat([train_df, test_df])
start_date = pd.Timestamp("2023-09-24")
plotdf = plotdf.with_columns(
    pl.from_epoch("first_submission_date", time_unit="s").alias("first_submission_dt")
)
plotdf = plotdf.with_columns(
    (
        (pl.col("first_submission_dt") - pl.lit(start_date)).dt.total_days() // 7
    ).cast(pl.Int64).alias("week")
)

print(plotdf.shape)

# Plot file types across weeks
gbdf = plotdf.group_by(["file_type", "week"]).agg(pl.len().alias("count"))
alt.Chart(gbdf).mark_bar().encode(
    alt.X('week:O', axis=alt.Axis(title='Week First Seen')),
    alt.Y('count:Q', axis=alt.Axis(title='File Type')),
        alt.Color('file_type:N', scale=alt.Scale(range=["#4c78a8", "#54a24b", "#f58518",  "#88d27a",  "#9ecae9", "#ffbf79"]),
              legend=alt.Legend(values=["Win32", "Win64", "Dot_Net", "APK", "ELF", "PDF"]))
)

(1344000, 16)


alt.Chart(...)